# Milestone 3 Exploration Notebook

This notebook contains supporting exploration and experiments for Milestone 3.

The full written discussion and final conclusions are provided in:
`results/milestone3_discussion.md`

## Step 1. Improve your Workflow 

### Dataset Scaling

**1. Setup and Load Retrieval Artifacts**

In [1]:
from pathlib import Path
import sys
import pandas as pd

# Project paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "processed_data_sample.parquet"
SEMANTIC_STORE_PATH = PROJECT_ROOT / "data" / "processed" / "faiss_store"
BM25_PATH = PROJECT_ROOT / "data" / "processed" / "bm25_retriever.pkl"

# Project imports
from src.semantic import get_or_build_vectorstore, get_semantic_retriever
from src.bm25 import load_bm25_retriever
from src.rag_pipeline import build_context, build_prompt, SYSTEM_PROMPT_FINAL


# Check required files
assert DATA_PATH.exists(), f"Missing parquet file: {DATA_PATH}"
assert SEMANTIC_STORE_PATH.exists(), f"Missing semantic store: {SEMANTIC_STORE_PATH}"
assert BM25_PATH.exists(), f"Missing BM25 retriever file: {BM25_PATH}"


# Load retrieval components
semantic_vectorstore = get_or_build_vectorstore(
    data_path=str(DATA_PATH),
    store_path=str(SEMANTIC_STORE_PATH),
    model_name="sentence-transformers/all-MiniLM-L6-v2",
)

semantic_retriever = get_semantic_retriever(semantic_vectorstore, top_k=5)
bm25 = load_bm25_retriever(str(BM25_PATH))

print("Semantic retriever ready.")
print("BM25 retriever loaded.")
print("Setup complete.")

/Users/clairesaunders/miniforge3/envs/retrieval-app/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 11307.47it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Semantic vector store already exists. Loading from disk.
Semantic retriever ready.
BM25 retriever loaded.
Setup complete.


**2. Check Dataset Scale Requirement (> 10,000 products)**

Since the original product identifier (`parent_asin`) was removed during preprocessing, we approximate the number of unique products using `product_title`. While this is not a perfect identifier, it provides a reasonable estimate of product-level uniqueness for the purpose of validating the dataset size.

In [2]:
df = pd.read_parquet(DATA_PATH)

n_rows = len(df)
n_unique_products = df["product_title"].nunique()

print(f"Rows in parquet: {n_rows:,}")
print(f"Approximate unique products (using product_title): {n_unique_products:,}")

Rows in parquet: 100,000
Approximate unique products (using product_title): 56,852


**3. Test Validity of Unique Product Title = Unique Product Assumption**

**Uniqueness Test 1:** To test the validity of unique product_title corresponding to unique products, we randomly sample 10 unique product titles and print out the products to see if any are duplicates. 

In [3]:
unique_products = df.drop_duplicates(subset=["product_title"])
sample_unique_products = unique_products.sample(n=100, random_state=42)
sample_unique_products[["product_title", "price", "average_rating"]]

,product_title,price,average_rating
73415,"Weststone 25pcs 2 1/2"" Solid Purple Satin Fabr...",8.95,5.0
15761,Char-Broil Surefire Electronic Ignition Button...,NaN,4.6
22062,Himal Outdoors Lawn Mower Cover -Tractor Cover...,24.99,4.6
66038,Tiemom Silicone Basting Brushes 3-Piece Set - Red,NaN,4.1
39814,"PURRUGS Dirt Trapper Door mat 24"" x 35.5"", Non...",34.99,4.5
...,...,...,...
63122,"EZGO Charger with 18-Feet Cord, 48-Volt",NaN,4.2
54896,Telescoping Green Windsock Pole - Heavy Duty -...,44.95,4.3
1590,Color-Changing Hot Tub Spa Light - 36 LEDs - V...,49.95,4.5
45932,Briggs & Stratton B4847GS Pressure Washer Quic...,24.95,4.7


**Uniqueness Test 2:** To verify that duplicate products have duplicate titles we run a semantic search query on 5 product titles and print out a sample of the returned products. 

In [4]:
sample_titles = unique_products["product_title"].sample(5, random_state=42)

In [5]:
for title in sample_titles:
    print("="*80)
    print("QUERY:", title, "\n")
    
    docs = semantic_retriever.invoke(title)
    
    for i, doc in enumerate(docs, 1):
        print(f"{i}. {doc.metadata.get('product_title')}")
    print("\n")

QUERY: Weststone 25pcs 2 1/2" Solid Purple Satin Fabric Pre-Tied Ribbon Bows for Cello Bags 

1. Weststone 25pcs 2 1/2" Solid Purple Satin Fabric Pre-Tied Ribbon Bows for Cello Bags
2. Weststone 100pcs 4" White Metallic Twist Ties foil Twist Ties for Cello Bags Treat Bags in Birthday Party Wedding Party
3. Weststone 200 pcs 4" Rainbow Color Metallic Twist Ties for Cello Bags in Birthday Party
4. Weststone 100pcs 4" White Metallic Twist Ties foil Twist Ties for Cello Bags Treat Bags in Birthday Party Wedding Party
5. Weststone 200 pcs 4" Rainbow Color Metallic Twist Ties for Cello Bags in Birthday Party


QUERY: Char-Broil Surefire Electronic Ignition Button , Black 

1. Char-Broil Surefire Electronic Ignition Button , Black
2. Char-Broil Surefire Electronic Ignition Button , Black
3. Char-Broil Surefire Electronic Ignition Button , Black
4. Char-Broil Surefire Electronic Ignition Button , Black
5. Char-Broil Surefire Electronic Ignition Button , Black


QUERY: Himal Outdoors Lawn Mower

### LLM Experiment

#### Why these two models?

We compare `llama-3.1-8b-instant` and `llama-3.3-70b-versatile` using the same retrieved context and the same final prompt.

- `llama-3.1-8b-instant` is the Milestone 2 baseline. It is fast, concise, and generally strong at following prompt constraints.
- `llama-3.3-70b-versatile` is a much larger model with stronger reasoning and generation capabilities. It is expected to produce more detailed and natural explanations, especially for queries that require synthesizing multiple retrieved documents.

This comparison helps evaluate how model scale and generation capability affect the quality of RAG outputs.

In [6]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os

load_dotenv()
assert os.getenv("GROQ_API_KEY") is not None, "Missing GROQ_API_KEY"

MODEL_1_NAME = "llama-3.1-8b-instant"
MODEL_2_NAME = "llama-3.3-70b-versatile"

llm_1 = ChatGroq(model=MODEL_1_NAME)
llm_2 = ChatGroq(model=MODEL_2_NAME)

print(f"Model 1 loaded: {MODEL_1_NAME}")
print(f"Model 2 loaded: {MODEL_2_NAME}")

Model 1 loaded: llama-3.1-8b-instant
Model 2 loaded: llama-3.3-70b-versatile


**Run 5 Queries on Both Models with Identical Context**

We evaluate both models on a set of representative queries to compare structure, relevance, and reasoning.

In [7]:
queries = [
    "6-inch ceramic plant pot with drainage hole",
    "blue patio chair cushion",
    "something to make a garden look more colorful and lively",
    "something to keep the deer out of my garden without hurting them",
    "what is a good plant to keep outside that won’t die if I forget to water it and doesn’t need much sunlight",
]

results = []

for query in queries:
    docs = semantic_retriever.invoke(query)
    context = build_context(docs)
    prompt = build_prompt(query, context, SYSTEM_PROMPT_FINAL)

    output_1 = llm_1.invoke(prompt).content
    output_2 = llm_2.invoke(prompt).content

    results.append({
        "query": query,
        "context": context,
        "model_1_name": MODEL_1_NAME,
        "model_2_name": MODEL_2_NAME,
        "model_1_output": output_1,
        "model_2_output": output_2,
    })

print("Comparison complete.")

Comparison complete.


In [8]:
for i, result in enumerate(results, start=1):
        print("=" * 100)
        print(f"QUERY {i}: {result['query']}")
        print("=" * 100)

        print(f"\n{result['model_1_name']}\n" + "-" * len(result['model_1_name']))
        print(result["model_1_output"])

        print(f"\n{result['model_2_name']}\n" + "-" * len(result['model_2_name']))
        print(result["model_2_output"])

        print("\n")


QUERY 1: 6-inch ceramic plant pot with drainage hole

llama-3.1-8b-instant
--------------------
**6-inch Ceramic Plant Pot with Drainage Hole**

1. **ADN Ceramic Planter, 6 Inch Plant Pots for Plants Modern Decor Indoor Flower Pot with Saucer and Drainage Hole, White** ($24.99)
The ADN Ceramic Planter fits the user's request with its 6-inch size and drainage hole. Reviewers have praised its modern design and functionality.

2. **Gohearin Ceramic Plant Flowerpot, Hand-Painted Flowerpot Set, 6.6+5.4-inch Indoor Flowerpot with Drain Hole, Used for Indoor Plants, Cactus, Succulent Plants and Other Outdoor Garden, 2-Piece Set** ($29.99)
The Gohearin Ceramic Plant Flowerpot has a drainage hole and comes in a 6.6-inch size, making it a suitable option for the user's request. Reviewers have appreciated its hand-painted design and functionality.

3. **YFFSRJDJ Ceramic Flower Plants Pots Planter with Drainage Hole, 6.0 inch+5.0 Inch. Indoor-Outdoor Large Round Succulent Orchid Pot Set (Blue)** (

**Test Model Timing**

In [9]:
import time

def timed_invoke(llm, prompt):
    """Invoke an LLM and return its response along with execution time."""
    start = time.time()
    response = llm.invoke(prompt).content
    end = time.time()
    return response, end - start

In [10]:
out1, t1 = timed_invoke(llm_1, prompt)
out2, t2 = timed_invoke(llm_2, prompt)

print(f"{MODEL_1_NAME} time: {t1:.2f}s")
print(f"{MODEL_2_NAME} time: {t2:.2f}s")

llama-3.1-8b-instant time: 10.43s
llama-3.3-70b-versatile time: 0.82s


**New Prompt Optimized for More Complex LLM**

Based on the observations above, we chose the more complex 70B model for its stronger reasoning, but refinied the prompt to retain the structured, easy-to-read format of the simpler model. This approach combines the strengths of both models. This is discussed in more detail in the `milestone3_discussion.md`. 

In [11]:
SYSTEM_PROMPT_OPTIMIZED = """
You are a helpful Amazon shopping assistant for Patio, Lawn and Garden products.

Answer the user's question using ONLY the provided Amazon product review and metadata context.
Do NOT make up products or details that are not supported by the context.
If the context is not sufficient, clearly say so.

FORMAT YOUR RESPONSE EXACTLY AS FOLLOWS:

- Provide up to 5 product recommendations when relevant.
- prioritize products that best match the user’s request
- use review insights to support explanations when helpful (e.g., durability, ease of use, common issues)
- avoid generic or repetitive statements about ratings (e.g., "highly rated", "4.5 stars") unless they add meaningful context
- do not include products that clearly do not match key constraints in the query (e.g., wrong color, size, or use)
- only include products that are supported by the context and are meaningfully relevant; do not include filler items to reach a specific number
- Use a clean numbered list (1., 2., 3., etc.) with no extra text between items
- Each item must follow this structure:

<Number>. **Product Title** (Price if available)
1–2 sentences of why this product fits the user’s request.

Optionally include a short one-line header before the list that directly reflects the user’s request.
Do NOT include long introductions or explanations before the list.
Do NOT include a concluding summary sentence.

STYLE GUIDELINES:
- Keep the tone professional and friendly
- Be concise and practical
- Avoid generic phrases (e.g., “great gift”) without specific reasoning
- Highlight what makes each option distinct when relevant
- Avoid duplicate or irrelevant products
"""

In [12]:
for query in queries:
    docs = semantic_retriever.invoke(query)
    context = build_context(docs)
    prompt = build_prompt(query, context, SYSTEM_PROMPT_OPTIMIZED)

    output = llm_2.invoke(prompt).content

    print("=" * 100)
    print(f"QUERY: {query}")
    print("=" * 100)
    print(output)
    print("\n")

QUERY: 6-inch ceramic plant pot with drainage hole
Ceramic Plant Pots with Drainage Holes
1. **ADN Ceramic Planter, 6 Inch Plant Pots for Plants Modern Decor Indoor Flower Pot with Saucer and Drainage Hole, White** ($24.99) - This product matches the user's request with its 6-inch size and drainage hole, making it a great option for indoor plants.
2. **Gohearin Ceramic Plant Flowerpot, Hand-Painted Flowerpot Set, 6.6+5.4-inch Indoor Flowerpot with Drain Hole** ($29.99) - Although slightly larger than 6 inches, this product has a drainage hole and can be used for indoor plants, making it a viable alternative.
3. **YFFSRJDJ Ceramic Flower Plants Pots Planter with Drainage Hole, 6.0 inch+5.0 Inch** ($26.99) - This set includes a 6-inch pot with a drainage hole, making it another option that meets the user's requirements.


QUERY: blue patio chair cushion
Patio Chair Cushions in Blue
1. **Rocking Chair Cushion Pad, 47IN Indoor Outdoor Navy Blue Rocking Chair Cushions Set** - This product m

## Step 2: Additional Feature (Option 3)

The product-level aggregation and scaling step was implemented separately because constructing the aggregated parquet required substantially more memory than was available on all development environments. We therefore kept the main milestone 3 notebook focused on analyses that can be run reproducibly from start to finish using the finalized processed dataset.

## Step 3: Improve Documentation and Code Quality

### Hybrid Retrieval Weight Tuning

#### Motivation

During initial testing of our hybrid retrieval system, we observed that the results were often nearly identical to those produced by semantic search alone. This suggested that the hybrid approach was not effectively incorporating the BM25 component, and that one retrieval method might be dominating the ensemble.

To better understand this behavior, we conducted a series of controlled experiments varying the semantic and BM25 weights. The goal was to determine whether different weighting schemes could produce a more balanced combination of keyword-based and semantic retrieval.

In [13]:
from langchain_classic.retrievers import EnsembleRetriever
from src.semantic import semantic_search
from src.bm25 import bm25_search

def make_hybrid_retriever(semantic_retriever, bm25_retriever, semantic_weight, bm25_weight):
    """Create a hybrid retriever combining semantic and BM25 with specified weights."""
    return EnsembleRetriever(
        retrievers=[semantic_retriever, bm25_retriever],
        weights=[semantic_weight, bm25_weight]
    )


def hybrid_titles(query, retriever, top_k=5):
    """Return top-k product titles from a retriever for a given query."""
    docs = retriever.invoke(query)
    return [doc.metadata.get("product_title", "") for doc in docs[:top_k]]


def overlap(a, b):
    """Return the number of shared elements between two lists."""
    return len(set(a) & set(b))

In [14]:
queries = [
    "blue patio chair cushion",
    "6-inch ceramic plant pot with drainage hole",
    "something to make a garden look more colorful and lively",
]

hybrid_64 = make_hybrid_retriever(semantic_retriever, bm25, 0.6, 0.4)
hybrid_55 = make_hybrid_retriever(semantic_retriever, bm25, 0.5, 0.5)
hybrid_46 = make_hybrid_retriever(semantic_retriever, bm25, 0.4, 0.6)

for query in queries:
    print("=" * 100)
    print(f"QUERY: {query}")
    print("=" * 100)

    semantic = [r["product_title"] for r in semantic_search(query, semantic_vectorstore, top_k=5)]
    bm25_res = [r["product_title"] for r in bm25_search(bm25, query, top_k=5)]

    h64 = hybrid_titles(query, hybrid_64)
    h55 = hybrid_titles(query, hybrid_55)
    h46 = hybrid_titles(query, hybrid_46)

    print("\nSEMANTIC:")
    for t in semantic:
        print("-", t)

    print("\nBM25:")
    for t in bm25_res:
        print("-", t)

    print("\nHYBRID (0.6 / 0.4):")
    for t in h64:
        print("-", t)

    print("\nHYBRID (0.5 / 0.5):")
    for t in h55:
        print("-", t)

    print("\nHYBRID (0.4 / 0.6):")
    for t in h46:
        print("-", t)

    print("\n--- OVERLAP ANALYSIS ---")
    print(f"0.6/0.4 overlap → semantic: {overlap(h64, semantic)}, bm25: {overlap(h64, bm25_res)}")
    print(f"0.5/0.5 overlap → semantic: {overlap(h55, semantic)}, bm25: {overlap(h55, bm25_res)}")
    print(f"0.4/0.6 overlap → semantic: {overlap(h46, semantic)}, bm25: {overlap(h46, bm25_res)}")


QUERY: blue patio chair cushion

SEMANTIC:
- Pillow Perfect Stripe Indoor/Outdoor 1 Piece Split Back Round Corner Chair Seat Cushion with Ties, Deep Seat, Weather, and Fade Resistant, 40.5" x 21", Red/White Midland, 1 Count
- Rocking Chair Cushion Pad, 47IN Indoor Outdoor Navy Blue Rocking Chair Cushions Set Soft Thickened Patio Chaise Lounger Cushion Overstuffed Patio Chair Cushion
- LVTXIII Outdoor Seat Cushions Patio Chair Pads with Ties, All Weather Chair Cushions for Home Office and Patio Garden Furniture Decoration D16”xW17”, Stripe Beige, Set of 2
- Indoor/Outdoor Bench Cushion Cotton Garden Furniture Loveseat Cushion, 51.2"x19.7" Patio Wicker Seat Cushions for Lounger Garden Furniture Patio Lounger Bench (Navy Blue)
- Indoor/Outdoor Bench Cushion, Swing Cushion, 51.2"x19.7", for Lounger Garden Furniture Patio Lounger Bench (Gray)

BM25:
- Rocking Chair Cushion Pad, 47IN Indoor Outdoor Navy Blue Rocking Chair Cushions Set Soft Thickened Patio Chaise Lounger Cushion Overstuffed P

## Final Choices

Based on the experiments in this notebook, we selected:
- `llama-3.3-70b-versatile` as the default LLM due to stronger reasoning and better handling of incomplete context
- a 0.5 / 0.5 hybrid weighting, since unequal weights caused the hybrid retriever to collapse too strongly toward either semantic or BM25 retrieval

These results informed the final discussion in `results/final_discussion.md`.